# TUNE

### IMPORTS

In [1]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.tsa.ar_model import AutoReg
from statsmodels.tsa.regime_switching.markov_regression import MarkovRegression
from sklearn.metrics import accuracy_score

np.random.seed(42)

# ------------------------------------------------------------
# 1. Simulate hidden Markov AR(1) spread
# ------------------------------------------------------------

N = 5000

# Transition matrix:
# P[i, j] = Prob(next state = j | current state = i)
P = np.array([
    [0.98, 0.02],   # calm regime is persistent
    [0.08, 0.92]    # turbulent regime is also persistent, but less so
])

mu_true = np.array([0.00, 0.00])
phi_true = np.array([0.25, 0.85])
sigma_true = np.array([0.50, 2.00])

states = np.zeros(N, dtype=int)
y = np.zeros(N)

for t in range(1, N):
    states[t] = np.random.choice([0, 1], p=P[states[t-1]])
    s = states[t]
    y[t] = mu_true[s] + phi_true[s] * y[t-1] + sigma_true[s] * np.random.randn()

# Drop burn-in
burn = 500
y = y[burn:]
states = states[burn:]

# ------------------------------------------------------------
# 2. Fit baseline AR(1)
# ------------------------------------------------------------

ar_model = AutoReg(y, lags=1, trend="c")
ar_res = ar_model.fit()

print("\n=== Baseline AR(1) ===")
print(ar_res.summary())

# ------------------------------------------------------------
# 3. Fit Markov-switching AR(1)
#    We manually include y_{t-1} as exogenous variable.
# ------------------------------------------------------------

y_dep = y[1:]
y_lag = y[:-1].reshape(-1, 1)
states_eval = states[1:]

ms_model = MarkovRegression(
    endog=y_dep,
    k_regimes=2,
    trend="c",
    exog=y_lag,
    switching_trend=True,
    switching_exog=True,
    switching_variance=True
)

ms_res = ms_model.fit(
    search_reps=20,
    em_iter=20,
    maxiter=500,
    disp=False
)

print("\n=== Markov-switching AR(1) ===")
print(ms_res.summary())

# ------------------------------------------------------------
# 4. Compare information criteria
# ------------------------------------------------------------

print("\n=== Model comparison ===")
print(f"AR(1) AIC:        {ar_res.aic:.2f}")
print(f"MS-AR(1) AIC:     {ms_res.aic:.2f}")
print(f"AR(1) BIC:        {ar_res.bic:.2f}")
print(f"MS-AR(1) BIC:     {ms_res.bic:.2f}")
print(f"AR(1) log-lik:    {ar_res.llf:.2f}")
print(f"MS-AR(1) log-lik: {ms_res.llf:.2f}")

# ------------------------------------------------------------
# 5. Regime classification accuracy
# ------------------------------------------------------------

probs = ms_res.smoothed_marginal_probabilities

# In statsmodels this may be a NumPy array or DataFrame depending on version
if isinstance(probs, pd.DataFrame):
    probs = probs.values

pred_states_raw = probs.argmax(axis=1)

# Markov states are label-invariant.
# State 0 and state 1 may be flipped relative to the simulation.
acc_1 = accuracy_score(states_eval, pred_states_raw)
acc_2 = accuracy_score(states_eval, 1 - pred_states_raw)

regime_accuracy = max(acc_1, acc_2)

print(f"\nRegime classification accuracy: {regime_accuracy:.3f}")

# ------------------------------------------------------------
# 6. Compare estimated regime persistence
# ------------------------------------------------------------

print("\nEstimated transition matrix:")
print(ms_res.regime_transition)


=== Baseline AR(1) ===
                            AutoReg Model Results                             
Dep. Variable:                      y   No. Observations:                 4500
Model:                     AutoReg(1)   Log Likelihood               -6408.325
Method:               Conditional MLE   S.D. of innovations              1.005
Date:                Sun, 03 May 2026   AIC                          12822.651
Time:                        22:49:20   BIC                          12841.886
Sample:                             1   HQIC                         12829.429
                                 4500                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0044      0.015      0.291      0.771      -0.025       0.034
y.L1           0.7348      0.010     72.679      0.000       0.715       0.755
                            